# Ch 4　用 LangGraph 思考：node、edge、state 三件事

> 這章重在「怎麼想」，所以 notebook 也輕量：我們把 Ch4 設計的 email agent **骨架**用不呼叫 LLM 的假節點搭起來，先把「形狀」跑出來、感受 node/edge/state 怎麼咬合。真正的 LLM 版在 Ch5。
>
> **這份 notebook 完全不需要 API key。**

In [ ]:
!uv pip install -q langgraph grandalf

## 把五步思考法的「圖」先用假節點搭出來

節點先全部用「貼標籤」的假實作，重點是看清楚：state 怎麼被一個個節點接力更新、分支怎麼走。

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

# state = agent 的「筆記本」。這裡放原始資料與中間產物，不放組好的文字（Ch4 的核心原則）。
class State(TypedDict):
    email: str                          # 原始信件（輸入）
    category: str                       # 分類結果（中間白板欄位）
    trace: Annotated[list[str], add]    # 走過哪些節點（累加，方便我們觀察）

def read_email(state):   return {"trace": ["read_email：把信讀進來"]}
def classify(state):
    # 假分類：含「壞掉/當機/504」就當複雜案件。Ch5 換成真的 LLM。
    complex_kw = any(k in state["email"] for k in ["壞", "當機", "504", "兩次"])
    return {"category": "complex" if complex_kw else "simple", "trace": ["classify：判斷意圖"]}
def draft(state):        return {"trace": ["draft：草擬回覆"]}
def escalate(state):     return {"trace": ["escalate：升級給真人"]}

def route(state):        # 邊只負責「決定往哪走」，不改 state
    return "escalate" if state["category"] == "complex" else "draft"

In [ ]:
b = StateGraph(State)
for name, fn in [("read_email", read_email), ("classify", classify),
                 ("draft", draft), ("escalate", escalate)]:
    b.add_node(name, fn)

b.add_edge(START, "read_email")
b.add_edge("read_email", "classify")
b.add_conditional_edges("classify", route, {"draft": "draft", "escalate": "escalate"})
b.add_edge("draft", END)
b.add_edge("escalate", END)
graph = b.compile()

# 簡單問題 -> 應該走 draft；複雜問題 -> 應該走 escalate。看 trace 就懂它走了哪條路。
print(graph.invoke({"email": "請問如何重設密碼？", "category": "", "trace": []})["trace"])
print(graph.invoke({"email": "匯出 PDF 會當機！", "category": "", "trace": []})["trace"])

看到了嗎？同一張圖，兩封信走了不同路徑——而「往哪走」的決定發生在節點/路由函數裡，不在箭頭本身。這就是 Ch4 想讓你內化的世界觀。下一章我們把假節點換成真的 LLM。

## 拓樸圖：眼見為憑

把剛剛建好的圖實際拓樸印出來，跟你腦中的形狀對照。

In [ ]:
print("--- ASCII ---")
print(graph.get_graph().draw_ascii())
print("\n--- Mermaid 原始碼 ---")
print(graph.get_graph().draw_mermaid())